# Parsing e análise de sentimentos em reviews da Steam

Este notebook adapta o fluxo de `PLN_2026_Aula_9_parsing,_NER_e_extração_de_relações.ipynb` ao conjunto `steam_reviews.pkl.gz`. A sequência é:

1. selecionar uma amostra pequena, balanceada e reproduzível de reviews em inglês;
2. aplicar o modelo inglês `en_core_web_sm` do spaCy para segmentação, classes gramaticais, lematização, dependências, entidades e relações simples;
3. aplicar VADER **à mesma amostra** para estimar sentimento;
4. comparar o sentimento textual com `voted_up`, entendido somente como um indicador de recomendação do usuário.

> O arquivo compactado é grande e precisa ser carregado integralmente pelo Pandas antes da amostragem. A primeira leitura pode demorar e consumir bastante memória.

## 1. Preparação do ambiente

O parsing usa o modelo em inglês porque as reviews selecionadas estão em inglês. VADER foi escolhido para sentimento por ser um método leve baseado em léxico, adequado a textos opinativos curtos e capaz de considerar intensificadores, negação, pontuação e emoticons.

In [ ]:
%pip -q install "pandas>=3.0,<4" "spacy>=3.8,<4" vaderSentiment nltk matplotlib

import gc
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import spacy
from IPython.display import display
from spacy import displacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("Pandas:", pd.__version__)
print("spaCy:", spacy.__version__, "| Modelo:", nlp.meta.get("name"), nlp.meta.get("version"))

## 2. Carregamento e seleção da amostra

A célula procura o arquivo tanto a partir da raiz do projeto quanto a partir da pasta `6_Parsing`. São selecionadas 15 reviews recomendadas e 15 não recomendadas, todas em inglês, com 10 a 80 palavras. A semente torna a seleção reproduzível.

In [ ]:
SEMENTE = 42
N_POR_CLASSE = 15

candidatos = [
    Path.cwd() / "_DadosLimpos" / "steam_reviews.pkl.gz",
    Path.cwd().parent / "_DadosLimpos" / "steam_reviews.pkl.gz",
]
arquivo_reviews = next((p for p in candidatos if p.exists()), None)
if arquivo_reviews is None:
    raise FileNotFoundError("Não foi possível localizar _DadosLimpos/steam_reviews.pkl.gz")

reviews = pd.read_pickle(arquivo_reviews)
colunas = [
    "appid", "game_name", "review_type", "voted_up",
    "review", "review_n_palavras", "review_idioma"
]
elegiveis = reviews.loc[
    reviews["review_idioma"].eq("en")
    & reviews["review"].notna()
    & reviews["review"].str.strip().ne("")
    & reviews["review_n_palavras"].between(10, 80),
    colunas,
].copy()
elegiveis["indice_fonte"] = elegiveis.index

positivas = elegiveis.loc[elegiveis["voted_up"]].sample(
    n=N_POR_CLASSE, random_state=SEMENTE
)
negativas = elegiveis.loc[~elegiveis["voted_up"]].sample(
    n=N_POR_CLASSE, random_state=SEMENTE
)
amostra = (
    pd.concat([positivas, negativas])
    .sample(frac=1, random_state=SEMENTE)
    .reset_index(drop=True)
)
amostra.insert(0, "review_id", range(1, len(amostra) + 1))

# Libera o DataFrame completo; as etapas seguintes usam apenas `amostra`.
del reviews, elegiveis, positivas, negativas
gc.collect()

print(f"Arquivo: {arquivo_reviews.resolve()}")
print(f"Amostra: {len(amostra)} reviews")
display(amostra["voted_up"].value_counts().rename(index={True: "recomendada", False: "não recomendada"}))
display(amostra[["review_id", "game_name", "voted_up", "review_n_palavras", "review"]])

## 3. Parsing com spaCy

Cada review é processada uma única vez com `nlp.pipe`. Os objetos `Doc` resultantes são reutilizados nas análises seguintes, garantindo que parsing, NER e relações usem exatamente os mesmos textos.

In [ ]:
docs = list(nlp.pipe(amostra["review"].tolist(), batch_size=16))
docs_por_id = dict(zip(amostra["review_id"], docs))

def tabela_dependencias(review_id):
    """Retorna tokens, lemas, classes e dependências de uma review da amostra."""
    doc = docs_por_id[review_id]
    return pd.DataFrame([
        {
            "review_id": review_id,
            "token": token.text,
            "lema": token.lemma_,
            "classe": token.pos_,
            "dependencia": token.dep_,
            "nucleo": token.head.text,
        }
        for token in doc
    ])

ID_EXEMPLO = int(amostra.iloc[12]["review_id"])
print(amostra.loc[amostra["review_id"].eq(ID_EXEMPLO), "review"].iloc[0])
display(tabela_dependencias(ID_EXEMPLO))

### 3.1. Visualização da árvore de dependências

A visualização mostra o núcleo de cada token e o rótulo da relação sintática. Reviews podem conter fragmentos, abreviações e erros de escrita; por isso, a análise deve ser interpretada como uma previsão do modelo.

In [ ]:
displacy.render(
    docs_por_id[ID_EXEMPLO],
    style="dep",
    jupyter=True,
    options={"compact": True, "distance": 90},
)

### 3.2. Resumo sintático de toda a amostra

A tabela resume, para cada review, o número de sentenças, tokens, substantivos, verbos e adjetivos. Isso transforma as saídas token a token em atributos comparáveis.

In [ ]:
resumo_parsing = pd.DataFrame([
    {
        "review_id": review_id,
        "sentencas": sum(1 for _ in doc.sents),
        "tokens": sum(not t.is_space for t in doc),
        "substantivos": sum(t.pos_ in {"NOUN", "PROPN"} for t in doc),
        "verbos": sum(t.pos_ in {"VERB", "AUX"} for t in doc),
        "adjetivos": sum(t.pos_ == "ADJ" for t in doc),
        "negacoes": sum(t.dep_ == "neg" for t in doc),
    }
    for review_id, doc in docs_por_id.items()
])
resumo_parsing = amostra[["review_id", "game_name", "voted_up"]].merge(
    resumo_parsing, on="review_id"
)
display(resumo_parsing)
display(resumo_parsing.drop(columns=["review_id", "game_name"]).groupby("voted_up").mean().round(2))

## 4. Entidades nomeadas e relações simples

Como no notebook de referência, primeiro extraímos entidades e depois usamos dependências para procurar relações aproximadas do tipo sujeito–verbo–objeto. Em reviews de jogos, nomes de jogos, empresas e personagens podem não ser reconhecidos corretamente pelo modelo geral `en_core_web_sm`.

In [ ]:
entidades = pd.DataFrame(
    [
        {
            "review_id": review_id,
            "entidade": ent.text,
            "categoria": ent.label_,
            "inicio": ent.start_char,
            "fim": ent.end_char,
        }
        for review_id, doc in docs_por_id.items()
        for ent in doc.ents
    ],
    columns=["review_id", "entidade", "categoria", "inicio", "fim"],
)
display(entidades if not entidades.empty else pd.DataFrame({"aviso": ["Nenhuma entidade reconhecida."]}))

displacy.render(docs_por_id[ID_EXEMPLO], style="ent", jupyter=True)

In [ ]:
def sintagma(token):
    """Expande o núcleo para sua subárvore, preservando a ordem original."""
    tokens = sorted((t for t in token.subtree if not t.is_punct), key=lambda t: t.i)
    return token.doc[tokens[0].i:tokens[-1].i + 1].text if tokens else token.text

def extrair_relacoes_doc(doc, review_id):
    """Extrai relações SVO e sujeito–verbo–preposição–objeto cobertas pelas regras."""
    relacoes = []
    for verbo in (t for t in doc if t.pos_ in {"VERB", "AUX"}):
        sujeitos = [t for t in verbo.children if t.dep_ in {"nsubj", "nsubjpass"}]
        objetos = [(t, verbo.lemma_) for t in verbo.children if t.dep_ in {"dobj", "obj", "attr", "oprd"}]
        for prep in (t for t in verbo.children if t.dep_ == "prep"):
            objetos.extend(
                (obj, f"{verbo.lemma_}_{prep.lemma_}")
                for obj in prep.children
                if obj.dep_ == "pobj"
            )
        for sujeito in sujeitos:
            for objeto, relacao in objetos:
                relacoes.append({
                    "review_id": review_id,
                    "sujeito": sintagma(sujeito),
                    "relacao": relacao,
                    "objeto": sintagma(objeto),
                })
    return relacoes

relacoes = pd.DataFrame(
    [r for review_id, doc in docs_por_id.items() for r in extrair_relacoes_doc(doc, review_id)],
    columns=["review_id", "sujeito", "relacao", "objeto"],
)
display(relacoes if not relacoes.empty else pd.DataFrame({"aviso": ["Nenhuma relação coberta pelas regras."]}))

## 5. Análise de sentimentos na mesma amostra

VADER devolve proporções negativa, neutra e positiva, além de `compound`, um escore normalizado entre −1 e 1. Usaremos os limiares recomendados pelo método: `compound >= 0.05` para positivo, `<= -0.05` para negativo e a faixa intermediária para neutro.

### 5.1. Textos completos usados na análise

Antes de calcular os sentimentos, exibimos abaixo o conteúdo integral das 30 reviews, junto com o jogo e a recomendação registrada na Steam.

In [ ]:
textos_completos = amostra[["review_id", "game_name", "voted_up", "review"]].copy()
textos_completos["recomendacao_steam"] = textos_completos["voted_up"].map(
    {True: "recomendada", False: "não recomendada"}
)
textos_completos = textos_completos.drop(columns="voted_up")

with pd.option_context(
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", None,
):
    display(textos_completos)

In [ ]:
analisador = SentimentIntensityAnalyzer()

def rotulo_sentimento(compound):
    if compound >= 0.05:
        return "positivo"
    if compound <= -0.05:
        return "negativo"
    return "neutro"

escores = amostra["review"].map(analisador.polarity_scores).apply(pd.Series)
resultado = pd.concat([amostra.reset_index(drop=True), escores], axis=1)
resultado["sentimento"] = resultado["compound"].map(rotulo_sentimento)
resultado["referencia_steam"] = resultado["voted_up"].map(
    {True: "positivo", False: "negativo"}
)
resultado["coincide_com_recomendacao"] = resultado["sentimento"].eq(
    resultado["referencia_steam"]
)

display(resultado[[
    "review_id", "game_name", "review", "referencia_steam",
    "neg", "neu", "pos", "compound", "sentimento",
    "coincide_com_recomendacao",
]])

### 5.2. Comparação com a recomendação da Steam

A recomendação binária e o sentimento textual medem conceitos relacionados, mas diferentes. Uma review pode elogiar aspectos de um jogo e ainda não recomendá-lo, usar sarcasmo ou relatar uma experiência mista. Assim, a comparação abaixo é uma avaliação exploratória, não uma validação com rótulos humanos de sentimento.

In [ ]:
matriz = pd.crosstab(
    resultado["referencia_steam"],
    resultado["sentimento"],
    rownames=["recomendação Steam"],
    colnames=["sentimento VADER"],
).reindex(index=["negativo", "positivo"], columns=["negativo", "neutro", "positivo"], fill_value=0)

cobertos = resultado["sentimento"].ne("neutro")
resumo_avaliacao = pd.Series({
    "n_reviews": len(resultado),
    "cobertura_nao_neutra": cobertos.mean(),
    "concordancia_global": resultado["coincide_com_recomendacao"].mean(),
    "concordancia_quando_nao_neutro": (
        resultado.loc[cobertos, "coincide_com_recomendacao"].mean()
        if cobertos.any() else float("nan")
    ),
})
display(matriz)
display(resumo_avaliacao.to_frame("valor").round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
resultado.boxplot(column="compound", by="referencia_steam", ax=axes[0])
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Escore VADER por recomendação")
axes[0].set_xlabel("Recomendação Steam")
axes[0].set_ylabel("compound")

imagem = axes[1].imshow(matriz.values, cmap="Blues")
axes[1].set_xticks(range(len(matriz.columns)), matriz.columns)
axes[1].set_yticks(range(len(matriz.index)), matriz.index)
axes[1].set_xlabel("Sentimento VADER")
axes[1].set_ylabel("Recomendação Steam")
axes[1].set_title("Tabela de contingência")
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        axes[1].text(j, i, matriz.iat[i, j], ha="center", va="center")
fig.colorbar(imagem, ax=axes[1], shrink=0.8)
fig.suptitle("")
plt.tight_layout()
plt.show()

#### Matriz de confusão binária

A matriz abaixo considera `referencia_steam` como classe real e `sentimento` como classe prevista. Como a Steam só fornece referência positiva ou negativa, resultados neutros do VADER são excluídos desta matriz e informados separadamente.

In [ ]:
avaliacao_binaria = resultado.loc[
    resultado["sentimento"].isin(["negativo", "positivo"])
].copy()
ordem_classes = ["negativo", "positivo"]
matriz_confusao = pd.crosstab(
    avaliacao_binaria["referencia_steam"],
    avaliacao_binaria["sentimento"],
    rownames=["classe real (Steam)"],
    colnames=["classe prevista (VADER)"],
).reindex(index=ordem_classes, columns=ordem_classes, fill_value=0)

vn = matriz_confusao.loc["negativo", "negativo"]
fp = matriz_confusao.loc["negativo", "positivo"]
fn = matriz_confusao.loc["positivo", "negativo"]
vp = matriz_confusao.loc["positivo", "positivo"]
total = matriz_confusao.to_numpy().sum()
dividir = lambda numerador, denominador: numerador / denominador if denominador else float("nan")
acuracia = dividir(vp + vn, total)
precisao = dividir(vp, vp + fp)
revocacao = dividir(vp, vp + fn)
f1 = dividir(2 * precisao * revocacao, precisao + revocacao)

metricas_binarias = pd.Series({
    "reviews avaliadas": total,
    "reviews neutras excluídas": resultado["sentimento"].eq("neutro").sum(),
    "acurácia": acuracia,
    "precisão (positivo)": precisao,
    "revocação (positivo)": revocacao,
    "F1 (positivo)": f1,
})
display(matriz_confusao)
display(metricas_binarias.to_frame("valor").round(3))

fig, ax = plt.subplots(figsize=(6, 5))
imagem = ax.imshow(matriz_confusao.values, cmap="Blues")
ax.set_xticks(range(2), ordem_classes)
ax.set_yticks(range(2), ordem_classes)
ax.set_xlabel("Classe prevista pelo VADER")
ax.set_ylabel("Classe real segundo a Steam")
ax.set_title("Matriz de confusão — sentimento × recomendação")
for i in range(2):
    for j in range(2):
        ax.text(j, i, matriz_confusao.iat[i, j], ha="center", va="center", fontsize=13)
fig.colorbar(imagem, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### 5.3. Inspeção dos casos divergentes

Os casos neutros ou divergentes são especialmente úteis para encontrar sarcasmo, negação complexa, vocabulário específico de jogos e diferença entre sentimento e intenção de recomendar.

In [ ]:
casos_para_inspecao = resultado.loc[
    ~resultado["coincide_com_recomendacao"],
    ["review_id", "game_name", "review", "referencia_steam", "sentimento", "compound"],
].sort_values("compound")
display(casos_para_inspecao)

## 6. Conclusões e limitações

- O spaCy converteu as reviews em estruturas sintáticas auditáveis: tokens, lemas, classes e dependências.
- NER e as relações baseadas em regras são aproximações; linguagem informal, sentenças incompletas e nomes específicos do universo dos jogos podem reduzir a qualidade.
- VADER não aprende com este conjunto e pode falhar com sarcasmo, gírias ou sentidos específicos de jogos.
- `voted_up` expressa recomendação, não uma anotação humana direta de sentimento. Divergências não são necessariamente erros.
- A amostra balanceada serve para demonstração e comparação qualitativa; ela não preserva a proporção original do conjunto nem sustenta conclusões sobre toda a base.

Como extensão, pode-se anotar manualmente uma amostra maior, calcular métricas contra esses rótulos e comparar VADER com um transformer treinado para sentimento em inglês.